# Testing ML Code with pytest

## Table of Contents

1. [Why Test ML Code?](#1)
2. [pytest Basics](#2)
3. [Fixtures: Reusable Test State](#3)
4. [pytest.mark.parametrize: Testing Many Inputs](#4)
5. [Testing ML-Specific Concerns](#5)
6. [Mocking External Dependencies](#6)
7. [Exercise: Test the Pipeline](#7)

---
## 1. Why Test ML Code? <a id="1"></a>

ML code is unusually easy to break silently. Consider:
- A scaler fitted on test data (data leakage) — model still trains, accuracy looks fine
- A label encoder whose `fit()` sees fewer categories than `transform()` — silent wrong encodings
- A pipeline that returns the wrong shape — downstream crash, not here

Tests catch these regressions before they reach production.

---
## 2. pytest Basics <a id="2"></a>

pytest collects and runs any function that starts with `test_` in files named `test_*.py`.

```bash
pytest src/tests/
pytest src/tests/test_data.py::test_dataset_split
pytest -v      # verbose output
pytest -x      # stop at first failure
```

For notebooks: you can run pytest tests as cells directly, or use the `nbmake` plugin.

In [ ]:
# Install pytest if needed
# !pip install pytest pytest-cov

import pytest
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


# --- The function under test ---
def calculate_class_balance(y: pd.Series) -> dict:
    """Return class counts and proportions."""
    counts = y.value_counts()
    props  = y.value_counts(normalize=True)
    return {"counts": counts.to_dict(), "proportions": props.to_dict()}


# --- Tests ---
def test_class_balance_basic():
    y = pd.Series([0, 0, 0, 1, 1])
    result = calculate_class_balance(y)
    assert result["counts"][0] == 3
    assert result["counts"][1] == 2
    assert abs(result["proportions"][0] - 0.6) < 1e-9

def test_class_balance_equal():
    y = pd.Series([0, 1, 0, 1])
    result = calculate_class_balance(y)
    assert result["proportions"][0] == result["proportions"][1]

def test_class_balance_all_same_class():
    y = pd.Series([1, 1, 1])
    result = calculate_class_balance(y)
    assert result["proportions"][1] == 1.0
    assert 0 not in result["counts"]


# Run inline (not how you'd do it in a real project — use pytest from terminal)
for fn in [test_class_balance_basic, test_class_balance_equal, test_class_balance_all_same_class]:
    fn()
    print(f"  {fn.__name__}: PASSED")

---
## 3. Fixtures: Reusable Test State <a id="3"></a>

Fixtures eliminate duplication in tests. Instead of building a dataset in every test, you define it once and inject it.

In [ ]:
# fixtures.py or conftest.py (pytest auto-discovers conftest.py)

import pytest
import pandas as pd
import numpy as np


@pytest.fixture
def small_dataset():
    """A tiny but realistic churn dataset for fast tests."""
    np.random.seed(42)
    n = 100
    X = pd.DataFrame({
        "tenure":  np.random.randint(1, 60, n),
        "charges": np.random.uniform(20, 100, n),
    })
    y = pd.Series(np.random.randint(0, 2, n), name="churn")
    return X, y


@pytest.fixture
def standard_scaler():
    """A fresh, unfitted StandardScaler."""
    # Import from your src package in a real project
    class StandardScaler:
        def __init__(self):
            self._mean = None
            self._std = None

        def fit(self, X):
            self._mean = X.mean()
            self._std = X.std().clip(lower=1e-8)
            return self

        def transform(self, X):
            return (X - self._mean) / self._std

        def fit_transform(self, X):
            return self.fit(X).transform(X)

    return StandardScaler()


# Using fixtures in tests
def test_scaler_zero_mean(small_dataset, standard_scaler):
    X, _ = small_dataset
    X_scaled = standard_scaler.fit_transform(X)
    means = X_scaled.mean()
    for col in X.columns:
        assert abs(means[col]) < 1e-9, f"Column {col} mean not zero after scaling"

def test_scaler_unit_variance(small_dataset, standard_scaler):
    X, _ = small_dataset
    X_scaled = standard_scaler.fit_transform(X)
    stds = X_scaled.std()
    for col in X.columns:
        assert abs(stds[col] - 1.0) < 1e-9, f"Column {col} std not 1 after scaling"

def test_scaler_requires_fit(standard_scaler):
    X = pd.DataFrame({"a": [1.0, 2.0, 3.0]})
    with pytest.raises(TypeError):
        standard_scaler.transform(X)  # should fail before fit


# Run inline
sc = standard_scaler()
ds = small_dataset()
for fn in [test_scaler_zero_mean, test_scaler_unit_variance, test_scaler_requires_fit]:
    try:
        fn(ds, sc) if fn != test_scaler_requires_fit else fn(sc)
        print(f"  {fn.__name__}: PASSED")
    except Exception as e:
        print(f"  {fn.__name__}: {type(e).__name__}: {e}")

---
## 4. pytest.mark.parametrize: Testing Many Inputs <a id="4"></a>

Instead of writing 5 separate tests for 5 inputs, parametrize one test.

In [ ]:
import pytest


def calculate_imbalance_ratio(y: pd.Series) -> float:
    """Return the ratio of majority to minority class."""
    counts = y.value_counts()
    return counts.iloc[0] / counts.iloc[-1]


# Instead of 4 separate test functions...
@pytest.mark.parametrize("counts, expected_ratio", [
    ([0, 0, 0, 1],           3.0),
    ([0, 0, 1, 1],           1.0),
    ([0]*9 + [1]*1,          9.0),
    ([0, 1, 1, 1, 1, 1, 1], 1/6),  # majority is class 1
])
def test_imbalance_ratio(counts, expected_ratio):
    y = pd.Series(counts)
    result = calculate_imbalance_ratio(y)
    assert abs(result - expected_ratio) < 1e-9


# Run all parametrized cases
test_cases = [
    ([0, 0, 0, 1],           3.0),
    ([0, 0, 1, 1],           1.0),
    ([0]*9 + [1]*1,          9.0),
    ([0, 1, 1, 1, 1, 1, 1], 1/6),
]
for counts, expected in test_cases:
    test_imbalance_ratio(counts, expected)
    print(f"  counts={counts[:4]}... expected={expected}: PASSED")

---
## 5. Testing ML-Specific Concerns <a id="5"></a>

ML pipelines have unique failure modes. These are the tests you should always write:

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split


def test_no_data_leakage():
    """Scaler must use only training statistics, not test statistics."""
    np.random.seed(42)
    X = pd.DataFrame({"a": np.random.randn(100), "b": np.random.randn(100)})
    y = pd.Series(np.random.randint(0, 2, 100))
    X_tr, X_te = X[:80], X[80:]

    class SimpleScaler:
        def fit(self, X):
            self._mean = X.mean()
            return self
        def transform(self, X):
            return X - self._mean

    scaler = SimpleScaler()
    scaler.fit(X_tr)
    X_te_scaled = scaler.transform(X_te)

    # Test: the scaler's mean came from train, not from test
    assert abs(scaler._mean["a"] - X_tr["a"].mean()) < 1e-9
    assert abs(scaler._mean["a"] - X_te["a"].mean()) > 0.01  # they should differ


def test_prediction_shape():
    """Model must return the correct output shape."""
    np.random.seed(0)
    X_tr = pd.DataFrame({"a": np.random.randn(80), "b": np.random.randn(80)})
    y_tr = pd.Series(np.random.randint(0, 2, 80))
    X_te = pd.DataFrame({"a": np.random.randn(20), "b": np.random.randn(20)})

    clf = RandomForestClassifier(n_estimators=10, random_state=0)
    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_te)

    assert preds.shape == (20,), f"Expected shape (20,), got {preds.shape}"
    assert set(preds).issubset({0, 1}), f"Predictions outside expected classes: {set(preds)}"


def test_predictions_in_valid_range():
    """predict_proba must return values in [0, 1] summing to 1."""
    np.random.seed(0)
    X = pd.DataFrame({"a": np.random.randn(100)})
    y = pd.Series(np.random.randint(0, 2, 100))

    clf = RandomForestClassifier(n_estimators=10, random_state=0)
    clf.fit(X, y)
    proba = clf.predict_proba(X)

    assert proba.min() >= 0.0, "Probabilities must be >= 0"
    assert proba.max() <= 1.0, "Probabilities must be <= 1"
    assert np.allclose(proba.sum(axis=1), 1.0), "Probabilities must sum to 1 per row"


def test_missing_value_handling():
    """Pipeline should handle NaN inputs gracefully, not silently."""
    X_with_nan = pd.DataFrame({"a": [1.0, np.nan, 3.0]})

    class StrictScaler:
        def fit(self, X):
            if X.isnull().any().any():
                raise ValueError("Training data contains NaN values.")
            self._mean = X.mean()
            return self
        def transform(self, X):
            return X - self._mean

    scaler = StrictScaler()
    with pytest.raises(ValueError, match="NaN"):
        scaler.fit(X_with_nan)


# Run all
for fn in [test_no_data_leakage, test_prediction_shape,
           test_predictions_in_valid_range, test_missing_value_handling]:
    fn()
    print(f"  {fn.__name__}: PASSED")

---
## 6. Mocking External Dependencies <a id="6"></a>

When a function calls a database, an API, or reads a file, you don't want tests to make real calls. `unittest.mock` lets you replace those calls with controlled fakes.

In [ ]:
from unittest.mock import patch, MagicMock
import pandas as pd


def load_from_database(table: str, connection_string: str) -> pd.DataFrame:
    """Load a table from a database."""
    import sqlite3
    conn = sqlite3.connect(connection_string)
    return pd.read_sql(f"SELECT * FROM {table}", conn)


# Test without actually touching a database
def test_load_uses_correct_table():
    mock_df = pd.DataFrame({"id": [1, 2], "churn": [0, 1]})

    with patch("pandas.read_sql", return_value=mock_df) as mock_read:
        with patch("sqlite3.connect") as mock_connect:
            result = load_from_database("customers", "fake_conn_string")

            # Assert the right table was queried
            call_args = mock_read.call_args
            assert "customers" in call_args[0][0]

            # Assert the result is what we injected
            assert len(result) == 2


test_load_uses_correct_table()
print("  test_load_uses_correct_table: PASSED")

---
## 7. Exercise: Test the Pipeline <a id="7"></a>

Write tests for a `ColumnDropper` transformer that removes specified columns.

**Requirements:**
1. `test_columns_are_removed`: verify target columns are gone from output
2. `test_other_columns_preserved`: verify non-target columns remain
3. `test_raises_on_missing_column`: verify it raises `KeyError` if a column to drop doesn't exist
4. `test_fit_transform_equivalent`: verify `fit_transform(X)` == `fit(X).transform(X)`

Bonus: use `@pytest.mark.parametrize` for test 1 and 2.

In [ ]:
import pytest
import pandas as pd
import numpy as np


class ColumnDropper:
    """Drops specified columns from a DataFrame."""

    def __init__(self, columns: list[str]):
        self.columns = columns

    def fit(self, X: pd.DataFrame) -> "ColumnDropper":
        missing = [c for c in self.columns if c not in X.columns]
        if missing:
            raise KeyError(f"Columns to drop not found: {missing}")
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return X.drop(columns=self.columns)

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return self.fit(X).transform(X)


# Your tests here
def test_columns_are_removed():
    pass  # implement me

def test_other_columns_preserved():
    pass  # implement me

def test_raises_on_missing_column():
    pass  # implement me

def test_fit_transform_equivalent():
    pass  # implement me


# Run
for fn in [test_columns_are_removed, test_other_columns_preserved,
           test_raises_on_missing_column, test_fit_transform_equivalent]:
    fn()
    print(f"  {fn.__name__}: PASSED")

> <details>
> <summary>Solution</summary>
>
> ```python
> def test_columns_are_removed():
>     X = pd.DataFrame({"a": [1, 2], "b": [3, 4], "c": [5, 6]})
>     result = ColumnDropper(["a", "b"]).fit_transform(X)
>     assert "a" not in result.columns
>     assert "b" not in result.columns
>
>
> def test_other_columns_preserved():
>     X = pd.DataFrame({"a": [1, 2], "b": [3, 4], "c": [5, 6]})
>     result = ColumnDropper(["a"]).fit_transform(X)
>     assert "b" in result.columns
>     assert "c" in result.columns
>
>
> def test_raises_on_missing_column():
>     X = pd.DataFrame({"a": [1, 2]})
>     with pytest.raises(KeyError):
>         ColumnDropper(["does_not_exist"]).fit(X)
>
>
> def test_fit_transform_equivalent():
>     X = pd.DataFrame({"a": [1.0, 2.0], "b": [3.0, 4.0]})
>     dropper = ColumnDropper(["a"])
>     result1 = dropper.fit_transform(X)
>     dropper2 = ColumnDropper(["a"])
>     result2 = dropper2.fit(X).transform(X)
>     pd.testing.assert_frame_equal(result1, result2)
> ```
> </details>